In [ ]:
%%sql -r dataframe_1
USE DATABASE AWSETL;
USE SCHEMA AWSETL;

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
df = session.table("AIR_CONDITIONERS")
df.show()

In [ ]:
import pandas as pd


In [ ]:
df.columns

In [ ]:
df.describe().show()

In [ ]:
df.describe().show()
df.select("MAIN_CATEGORY").distinct().show()
df.select("SUB_CATEGORY").distinct().show()

In [ ]:
from snowflake.snowpark.functions import col, call_builtin

df = df.with_column(
    "RATINGS_CLEAN",
    call_builtin("TRY_TO_DOUBLE", col("RATINGS"))
)

df = df.filter(col("RATINGS_CLEAN").is_not_null())

In [ ]:
df.select("RATINGS", "RATINGS_CLEAN").limit(20).show()

In [ ]:
from snowflake.snowpark.functions import col, sql_expr

df = df.with_column(
    "REVIEWS_CLEAN",
    sql_expr("TRY_TO_NUMBER(REPLACE(REGEXP_SUBSTR(NO_OF_RATINGS, '[0-9,]+'), ',', ''))")
)

In [ ]:
from snowflake.snowpark.functions import replace

df = df.with_column(
    "REVIEWS_CLEAN",
    sql_expr("TRY_TO_NUMBER(REPLACE(REVIEWS_CLEAN, ',', ''))")
)

In [ ]:
df.select("NO_OF_RATINGS", "REVIEWS_CLEAN").limit(20).show()

In [ ]:
from snowflake.snowpark.functions import col

df = df.filter(col("ACTUAL_PRICE_CLEAN").is_not_null())
df = df.filter(col("DISCOUNT_PRICE_CLEAN").is_not_null())

In [ ]:
from snowflake.snowpark.functions import sql_expr

df = df.with_column(
    "DISCOUNT_PERCENT",
    sql_expr("((ACTUAL_PRICE_CLEAN - DISCOUNT_PRICE_CLEAN) / ACTUAL_PRICE_CLEAN) * 100")
)

In [ ]:
df = df.with_column(
    "BRAND",
    sql_expr("SPLIT_PART(NAME, ' ', 1)")
)

In [ ]:
df.select(
    "NAME",
    "BRAND",
    "RATINGS_CLEAN",
    "REVIEWS_CLEAN",
    "ACTUAL_PRICE_CLEAN",
    "DISCOUNT_PRICE_CLEAN",
    "DISCOUNT_PERCENT"
).limit(10).show()

In [ ]:
df = df.with_column(
    "TON",
    sql_expr("TRY_TO_DOUBLE(REGEXP_SUBSTR(NAME, '[0-9]+\\.?[0-9]*\\s*Ton'))")
)

In [ ]:
df = df.with_column(
    "TON",
    sql_expr("TRY_TO_DOUBLE(REGEXP_SUBSTR(NAME, '[0-9]+\\.?[0-9]*'))")
)

In [ ]:
df = df.with_column(
    "STAR_RATING",
    sql_expr("TRY_TO_NUMBER(REGEXP_SUBSTR(NAME, '[3-5]\\s*Star'))")
)

In [ ]:
df = df.with_column(
    "AC_TYPE",
    sql_expr("""
    CASE
        WHEN LOWER(NAME) LIKE '%split%' THEN 'Split'
        WHEN LOWER(NAME) LIKE '%window%' THEN 'Window'
        ELSE 'Other'
    END
    """)
)

In [ ]:
df.select(
    "NAME",
    "BRAND",
    "TON",
    "STAR_RATING",
    "AC_TYPE",
    "RATINGS_CLEAN",
    "REVIEWS_CLEAN",
    "DISCOUNT_PERCENT"
).limit(10).show()

In [ ]:
df = df.with_column(
    "STAR_RATING",
    sql_expr("TRY_TO_NUMBER(REGEXP_SUBSTR(NAME, '[3-5](?=\\s*Star)'))")
)

In [ ]:
from snowflake.snowpark.functions import sql_expr

df = df.with_column(
    "STAR_RATING",
    sql_expr("TRY_TO_NUMBER(RIGHT(SPLIT_PART(NAME, ' Star', 1), 1))")
)

In [ ]:
df.select("NAME", "STAR_RATING").limit(10).show()

In [ ]:
pdf = df.to_pandas()

In [ ]:
type(df)

In [ ]:
type(pdf)

In [ ]:
features = pdf[[
    "TON",
    "STAR_RATING",
    "RATINGS_CLEAN",
    "REVIEWS_CLEAN",
    "ACTUAL_PRICE_CLEAN",
    "DISCOUNT_PRICE_CLEAN",
    "DISCOUNT_PERCENT"
]]

In [ ]:
features = features.dropna()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(features)

In [ ]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)

model.fit(X)

In [ ]:
features["ANOMALY"] = model.predict(X)

In [ ]:
features["ANOMALY"].value_counts()

In [ ]:
anomalies = pdf.loc[features.index]
anomalies["ANOMALY"] = features["ANOMALY"]

anomalies[anomalies["ANOMALY"] == -1][[
    "NAME",
    "BRAND",
    "TON",
    "STAR_RATING",
    "RATINGS_CLEAN",
    "REVIEWS_CLEAN",
    "ACTUAL_PRICE_CLEAN",
    "DISCOUNT_PRICE_CLEAN",
    "DISCOUNT_PERCENT"
]]

In [ ]:
import matplotlib.pyplot as plt

# attach anomaly labels back to dataframe
pdf["ANOMALY"] = features["ANOMALY"]

# scatter plot
plt.scatter(pdf["ACTUAL_PRICE_CLEAN"], pdf["RATINGS_CLEAN"])

# highlight anomalies
anomaly_points = pdf[pdf["ANOMALY"] == -1]
plt.scatter(anomaly_points["ACTUAL_PRICE_CLEAN"], anomaly_points["RATINGS_CLEAN"])

plt.xlabel("Actual Price")
plt.ylabel("Rating")
plt.title("Price vs Rating with Detected Anomalies")

plt.show()